# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rupikaupa2006/Flyrank-ML-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [14]:
%pip -q install duckdb huggingface_hub

In [15]:
import os, getpass

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

In [16]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:22} {n:>12,} rows")

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*


The ranked action queue prioritizes content that is most likely to benefit from optimization. Each content item receives a recommended action and a reason code explaining why it appears in the queue.

| Priority | Action | Reason Code |
|----------|--------|-------------|
| High | Refresh content | Low search impressions with high search potential |
| High | Improve SEO | Low click-through performance |
| Medium | Monitor performance | Stable performance with room for improvement |
| Medium | Update keywords | Competition is high or search intent may have changed |
| Low | No action | Content is already performing well |

The action queue is intended to support content teams by highlighting which pages deserve attention first.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


import pandas as pd

action_queue = con.sql(f"""
SELECT
    content_hash_id,
    client_hash_id,
    search_volume,
    competition,
    word_count
FROM {TABLES['dim_content']}
WHERE is_published = TRUE
LIMIT 20
""").df()

def recommend_action(row):
    if pd.notna(row["search_volume"]) and row["search_volume"] > 100:
        return "Refresh Content", "HIGH_SEARCH_VOLUME"
    elif pd.notna(row["competition"]) and row["competition"] > 0.7:
        return "Improve SEO", "HIGH_COMPETITION"
    else:
        return "Monitor", "STABLE_CONTENT"

actions = action_queue.apply(recommend_action, axis=1)

action_queue["Action"] = [a[0] for a in actions]
action_queue["Reason Code"] = [a[1] for a in actions]

action_queue

,content_hash_id,client_hash_id,search_volume,competition,word_count,Action,Reason Code
0,content_004de9653278b5a4,client_04660893ae39614a,30,0.91,2555,Improve SEO,HIGH_COMPETITION
1,content_00dc5efae381b2ab,client_04660893ae39614a,10,0.00,2430,Monitor,STABLE_CONTENT
2,content_01410f2556c327ac,client_04660893ae39614a,480,0.36,2645,Refresh Content,HIGH_SEARCH_VOLUME
3,content_019f27f634053ca7,client_04660893ae39614a,0,0.00,2522,Monitor,STABLE_CONTENT
4,content_01efa71faea45dcc,client_04660893ae39614a,2400,0.70,2552,Refresh Content,HIGH_SEARCH_VOLUME
5,content_01fc9e2e57898b55,client_04660893ae39614a,260,0.14,2672,Refresh Content,HIGH_SEARCH_VOLUME
6,content_0212158fa61c5fcb,client_04660893ae39614a,10,0.00,2396,Monitor,STABLE_CONTENT
7,content_023d807c7d922db1,client_04660893ae39614a,10,0.57,1929,Monitor,STABLE_CONTENT
8,content_026f7405cc253242,client_04660893ae39614a,4400,0.80,2481,Refresh Content,HIGH_SEARCH_VOLUME
9,content_02c8b23ea5bdb275,client_04660893ae39614a,390,0.45,2357,Refresh Content,HIGH_SEARCH_VOLUME


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*



This action playbook is designed to support content teams in prioritizing content optimization efforts. The recommendations are based on observed search performance and content characteristics.

The recommendations should not be treated as automatic decisions. Business priorities, seasonal trends, editorial judgment, and recent content updates are not represented in the available data. Human review is therefore required before implementing any recommendation.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Intended users: SEO teams and content managers")
print("Purpose: Prioritize content optimization")
print("Limitation: Recommendations require human review")

Intended users: SEO teams and content managers
Purpose: Prioritize content optimization
Limitation: Recommendations require human review


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*



Before acting on any recommendation, a content reviewer should verify:

- Whether the page has been recently updated.
- Whether business priorities have changed.
- Whether seasonal effects explain the observed performance.
- Whether technical issues affected search performance.

The following actions should never be automated:

- Publishing or deleting content.
- Major content rewrites.
- Business-critical SEO decisions.
- Changes affecting legal or compliance content.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
review = pd.DataFrame({
    "Human Review Required": [
        "Recent updates",
        "Business priority",
        "Seasonality",
        "Technical issues"
    ]
})

review

,Human Review Required
0,Recent updates
1,Business priority
2,Seasonality
3,Technical issues


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*



The recommendations should be monitored over time.

The model should be retrained if:

- Search traffic changes significantly.
- New content types are introduced.
- Search engine algorithms change.
- Prediction performance decreases.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Retrain Trigger Checklist")
print("-------------------------")
print("Large traffic changes")
print("New content")
print("Algorithm updates")
print("Performance degradation")

Retrain Trigger Checklist
-------------------------
Large traffic changes
New content
Algorithm updates
Performance degradation


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*



The ranked action queue is exported as a CSV file so it can be reused in the final research paper and future analysis.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

os.makedirs("work/outputs", exist_ok=True)

action_queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Export completed successfully!")

action_queue.head()

Export completed successfully!


,content_hash_id,client_hash_id,search_volume,competition,word_count,Action,Reason Code
0,content_004de9653278b5a4,client_04660893ae39614a,30,0.91,2555,Improve SEO,HIGH_COMPETITION
1,content_00dc5efae381b2ab,client_04660893ae39614a,10,0.00,2430,Monitor,STABLE_CONTENT
2,content_01410f2556c327ac,client_04660893ae39614a,480,0.36,2645,Refresh Content,HIGH_SEARCH_VOLUME
3,content_019f27f634053ca7,client_04660893ae39614a,0,0.00,2522,Monitor,STABLE_CONTENT
4,content_01efa71faea45dcc,client_04660893ae39614a,2400,0.70,2552,Refresh Content,HIGH_SEARCH_VOLUME


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.